# Week 2 ID2221

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.window import Window
import pyspark.sql.functions as F

#The second line (with 4g) specifies how much RAM to use. change according to machine
spark = SparkSession.builder \
    .appName("IngestionFramework") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [2]:
integrated_taxi_trips = spark.read.format("delta").load("delta/integrated_taxi_trips_by_borough")

26/09/16 15:38:12 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


1. Monthly taxi demand for each taxi zone

In [ ]:
result = (integrated_taxi_trips
          .select("pu_zone", "tpep_pickup_datetime")
          .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
          .groupBy(["pu_zone", "month"])
          .count())

# Uncomment this line if you are doing latency measurements
# By default, spark is lazy and only calculates when the results are requested

# result.collect()

result.show(5)

26/09/15 13:47:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+-------------------+------+
|             pu_zone|              month| count|
+--------------------+-------------------+------+
|Upper West Side S...|2024-03-01 00:00:00|102427|
|Washington Height...|2024-01-01 00:00:00|   495|
|                SoHo|2024-03-01 00:00:00| 28612|
|      Yorkville East|2024-03-01 00:00:00| 48901|
|Upper West Side S...|2024-02-01 00:00:00| 90849|
+--------------------+-------------------+------+
only showing top 5 rows



2. Average trip distance under different weather conditions

In [ ]:
# Here we look at the average trip distance if there is rain, or no rain.

result = (
    integrated_taxi_trips
          .select("trip_distance", "weather_prcp")
          .dropna()
          .withColumn("is_raining", F.col("weather_prcp") != 0.0)
          .groupBy("is_raining")
          .avg("trip_distance")
          .withColumn("avg_trip_distance", F.round("avg(trip_distance)", 2))
          .drop("weather_prcp", "avg(trip_distance)")
    )

result.show()

26/09/16 15:38:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+-----------------+
|is_raining|avg_trip_distance|
+----------+-----------------+
|      true|              4.6|
|     false|             4.68|
+----------+-----------------+



3. Relationship between air quality and taxi demand

In [ ]:
# Get the hourly taxi demand associated to the average air quality measurement
result = (
    integrated_taxi_trips
        .select("tpep_pickup_datetime","air_q_sample_measurement")
        # A bunch of lines do not have any air quality data, they are removed
        .dropna()
        .withColumn("hour", F.date_trunc("hour", "tpep_pickup_datetime"))
        .groupBy("hour")
        .agg(
            F.count("*").alias("taxi_demand"),
            F.avg("air_q_sample_measurement").alias("air_quality")
        )
)

# Calculate the correlation between air quality and taxi demand
correlation = result.stat.corr(
    "air_quality",
    "taxi_demand"
)

print(correlation) # Result: corr = -0.017 -> No correlation 

-0.016985133728975584


4. Taxi zones with the largest variation in demand under different weather conditions

In [ ]:
result = (
    integrated_taxi_trips
        .select("pu_zones", "weather_prcp")
        .dropna()
        .withColumn("is_raining", F.col("weather_prcp") != 0.0)
        .withColumn("")
)

5. Peak travel hours for each day of the week

In [ ]:
result = (
    integrated_taxi_trips
    .select("tpep_pickup_datetime")
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week", "hour")
    .count()
)

# The window is used to get the max count for each day while keeping other informations (here, the busiest hour)
window = Window.partitionBy("day_of_week").orderBy(F.desc("count"))

result = (
    result
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") == 1)
    .drop("rank")
)


result.show()

+-----------+----+------+
|day_of_week|hour| count|
+-----------+----+------+
|     Friday|  18|110876|
|     Monday|  18| 87453|
|   Saturday|  19|103639|
|     Sunday|   0| 84325|
|   Thursday|  18|126855|
|    Tuesday|  18|106166|
|  Wednesday|  18|117895|
+-----------+----+------+



6. Monthly trends in taxi demand

In [ ]:
result = (integrated_taxi_trips
    .select("tpep_pickup_datetime")
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
    .groupBy("month")
    .count()
 )

window = Window.orderBy("month")

# Calculate the difference of number of trips between the current month and the previous month
result = (
    result
    .withColumn("previous_count", F.lag("count").over(window))
    .withColumn("difference", F.expr("count - previous_count"))
)

result.show()

26/09/16 16:49:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/16 16:49:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/16 16:49:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/16 16:49:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/16 16:49:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/16 16:49:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/16 1

+-------------------+-------+--------------+----------+
|              month|  count|previous_count|difference|
+-------------------+-------+--------------+----------+
|2002-12-01 00:00:00|      4|          NULL|      NULL|
|2008-12-01 00:00:00|      1|             4|        -3|
|2009-01-01 00:00:00|      4|             1|         3|
|2023-12-01 00:00:00|     10|             4|         6|
|2024-01-01 00:00:00|3073690|            10|   3073680|
|2024-02-01 00:00:00|3263634|       3073690|    189944|
|2024-03-01 00:00:00|3913076|       3263634|    649442|
|2024-04-01 00:00:00|      3|       3913076|  -3913073|
+-------------------+-------+--------------+----------+

